# 02 — Analyze a Recording

Record a new audio clip from the microphone, run the full analysis pipeline
(pitch detection → stability metrics → register analysis), display all plots,
and print a summary of the results.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt

from audio.recorder import record_audio
from audio.pitch_detector import detect_pitch, get_note_name, hz_to_cents
from analysis.stability import (
    compute_stability_metrics,
    identify_unstable_regions,
    register_analysis,
)
from visualization.plotter import (
    plot_pitch_contour,
    plot_stability_heatmap,
    plot_register_comparison,
)
from utils.config import SAMPLE_RATE

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 1. Record audio

Set `DURATION` to however many seconds you want to record.  
Set `SAVE_AS` to a filename (without extension) to keep the WAV, or `None` to discard it.

In [ ]:
DURATION = 10       # seconds
SAVE_AS  = None     # e.g. 'bb3_long_tone'

audio = record_audio(duration=DURATION, sample_rate=SAMPLE_RATE, filename=SAVE_AS)
print(f'Captured {len(audio)} samples  ({len(audio)/SAMPLE_RATE:.2f}s)')

## 2. Pitch detection

In [ ]:
times, frequencies, confidence = detect_pitch(audio, sr=SAMPLE_RATE)

voiced_mask = ~np.isnan(frequencies)
print(f'Voiced frames : {voiced_mask.sum()} / {len(times)}  ({voiced_mask.mean()*100:.1f}%)')

if voiced_mask.any():
    mean_hz = float(np.nanmean(frequencies))
    print(f'Mean pitch    : {mean_hz:.2f} Hz  →  {get_note_name(mean_hz)}  '
          f'({hz_to_cents(mean_hz):+.1f} cents from A4)')

## 3. Pitch contour

In [ ]:
fig = plot_pitch_contour(times, frequencies, title='Pitch Contour', save=True)
plt.show()

## 4. Stability heatmap

In [ ]:
fig = plot_stability_heatmap(times, frequencies, window_size=1.0, save=True)
plt.show()

## 5. Register comparison

In [ ]:
reg_metrics = register_analysis(times, frequencies)
fig = plot_register_comparison(reg_metrics, save=True)
plt.show()

## 6. Unstable regions

In [ ]:
regions = identify_unstable_regions(times, frequencies, threshold=20)

if not regions:
    print('No unstable regions detected — great intonation!')
else:
    print(f'Found {len(regions)} unstable region(s):\n')
    print(f'  {"Start":>7}  {"End":>7}  {"Note":>6}  {"Variance (cents²)":>18}')
    print('  ' + '-' * 44)
    for start, end, note, var in regions:
        print(f'  {start:7.2f}s  {end:7.2f}s  {note:>6}  {var:18.1f}')

## 7. Per-register summary

In [ ]:
for register, data in reg_metrics.items():
    print(f'\n── {register.upper()} register ──')
    print(f'  Frames       : {data["frame_count"]}')
    print(f'  Time fraction: {data["time_fraction"]*100:.1f}%')
    m = data['metrics']
    if m is not None:
        valid_scores = m['stability_score'][~np.isnan(m['stability_score'])]
        if valid_scores.size > 0:
            print(f'  Mean stability score : {valid_scores.mean():.3f}')
            print(f'  Min stability score  : {valid_scores.min():.3f}')
    print(f'  Unstable regions : {len(data["unstable_regions"])}')